In [1]:
pip install pymongo pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.6 MB/s eta 0:00:00


In [2]:
pip install pymongo certifi

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf # Data from Yahoo Finance
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from pymongo import MongoClient, UpdateOne
import certifi

# ETL

In [4]:
def ETL():

    # --- DOWNLOAD ---
    tickers = {
        "SP500": "^GSPC",
        "VIX": "^VIX",
        "MOVE": "^MOVE",
        "VIX3M": "^VIX3M",
        "DXY": "DX-Y.NYB",
        "GOLD": "GC=F",
        "OIL": "CL=F",
        "HYG": "HYG",
        "LQD": "LQD"
    }

    data = {}
    for name, ticker in tickers.items():
        df = yf.Ticker(ticker).history(period="20y")

        df = df.drop(columns=["Dividends", "Stock Splits"], errors="ignore")

        if name != "SP500":
            df = df.drop(columns=["Volume"], errors="ignore")

        df = df.rename(columns={
            "Open": f"Open_{name}",
            "High": f"High_{name}",
            "Low": f"Low_{name}",
            "Close": f"Close_{name}",
            "Volume": f"Volume_{name}"
        })

        df.index = df.index.tz_localize(None)
        data[name] = df

    # --- MERGE + ALIGNMENT ---
    dataset = pd.concat(data.values(), axis=1).sort_index()
    dataset = dataset.asfreq("B")
    dataset = dataset.ffill()

    # --- SHIFTED SERIES (ANTI-LEAKAGE CORE) ---
    close_spx = dataset["Close_SP500"].shift(1)
    close_vix = dataset["Close_VIX"].shift(1)
    close_move = dataset["Close_MOVE"].shift(1)
    close_vix3m = dataset["Close_VIX3M"].shift(1)

    return_spx = close_spx.pct_change()
    return_vix = close_vix.pct_change()
    return_move = close_move.pct_change()
    return_vix3m = close_vix3m.pct_change()

    # --- RETURNS ---
    dataset["Return_SPX"]  = return_spx
    dataset["Return_VIX"]  = return_vix
    dataset["Return_MOVE"] = return_move
    dataset["Return_VIX3M"]= return_vix3m

    # --- VOL ---
    dataset["RV_5d"]  = return_spx.rolling(5).std() * np.sqrt(252)
    dataset["RV_10d"] = return_spx.rolling(10).std() * np.sqrt(252)
    dataset["RV_21d"] = return_spx.rolling(21).std() * np.sqrt(252)

    dataset["VIX_Vol_5d"]  = return_vix.rolling(5).std()
    dataset["VIX_Vol_10d"] = return_vix.rolling(10).std()
    dataset["VIX_Vol_21d"] = return_vix.rolling(21).std()

    # --- LAGS ---
    dataset["VIX_Lag1"] = dataset["Close_VIX"].shift(1)
    dataset["VIX_Lag2"] = dataset["Close_VIX"].shift(2)
    dataset["VIX_Lag5"] = dataset["Close_VIX"].shift(5)

    # --- MOVING STATS (NO LEAKAGE focused) ---
    dataset["VIX_MA_5"]  = close_vix.rolling(5).mean()
    dataset["VIX_MA_10"] = close_vix.rolling(10).mean()
    dataset["VIX_MA_20"] = close_vix.rolling(20).mean()

    dataset["VIX_STD_5"]  = close_vix.rolling(5).std()
    dataset["VIX_STD_10"] = close_vix.rolling(10).std()
    dataset["VIX_STD_20"] = close_vix.rolling(20).std()

    dataset["VIX_Percentile"] = close_vix.rolling(252).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1]
    )

    # --- VOLUME ---
    dataset["SPX_Volume_Norm"] = dataset["Volume_SP500"] / (
        dataset["Volume_SP500"].rolling(252).mean() + 1e-8
    )

    # --- STRUCTURE ---
    dataset["VIX3M_Spread"] = close_vix - close_vix3m
    dataset["VIX_Contango"] = close_vix3m / (close_vix + 1e-8) - 1

    # --- GAPS ---
    dataset["SPX_Gap"] = (dataset["Open_SP500"] - close_spx) / (close_spx + 1e-8)
    dataset["VIX_Gap"] = (dataset["Open_VIX"] - close_vix) / (close_vix + 1e-8)

    # --- TREND / MOMENTUM ---
    dataset["Drawdown"] = close_spx / close_spx.cummax() - 1

    dataset["Momentum_1M"] = close_spx / close_spx.shift(21) - 1
    dataset["Momentum_3M"] = close_spx / close_spx.shift(63) - 1
    dataset["Momentum_6M"] = close_spx / close_spx.shift(126) - 1

    dataset["VIX_Zscore"] = (
        close_vix - dataset["VIX_MA_20"]
    ) / (dataset["VIX_STD_20"] + 1e-8)

    dataset["VIX_MeanRev"] = close_vix - dataset["VIX_MA_10"]

    dataset["IV_RV_Ratio"] = close_vix / (dataset["RV_21d"] + 1e-8)
    dataset["VIX_RV_Spread"] = close_vix - dataset["RV_21d"]

    dataset["VIX_Trend"] = (
        close_vix.ewm(span=21, adjust=False).mean()
        - close_vix.ewm(span=63, adjust=False).mean()
    )

    dataset["VIX_MOVE_Ratio"] = close_vix / (close_move + 1e-8)

    dataset["SPX_VIX_Corr_21d"] = return_spx.rolling(21).corr(return_vix)

    dataset["RV_21d_Sq"] = dataset["RV_21d"] ** 2
    dataset["VIX_Zscore_Sq"] = dataset["VIX_Zscore"] ** 2

    # --- MACRO FEATURES (NO LEAKAGE focus) ---
    dataset["DXY_overnight"]  = dataset["Open_DXY"]  / dataset["Close_DXY"].shift(1)  - 1
    dataset["GOLD_overnight"] = dataset["Open_GOLD"] / dataset["Close_GOLD"].shift(1) - 1
    dataset["OIL_overnight"]  = dataset["Open_OIL"]  / dataset["Close_OIL"].shift(1)  - 1

    # --- TARGET (categorical and balanced q1,q2,q3) ---
    dataset["Intraday_VIX_Return"] = (
        dataset["Close_VIX"] - dataset["Open_VIX"]
    ) / (dataset["Open_VIX"] + 1e-8)

    dataset["q_up"] = dataset["Intraday_VIX_Return"].shift(1).rolling(252).quantile(0.66)
    dataset["q_down"] = dataset["Intraday_VIX_Return"].shift(1).rolling(252).quantile(0.33)

    dataset["Intraday_VIX_Move"] = np.where(
        dataset["Intraday_VIX_Return"] >= dataset["q_up"], 1,
        np.where(dataset["Intraday_VIX_Return"] <= dataset["q_down"], 2, 0)
    )

    # --- FEATURES ---
    feature_cols = [
        "Open_SP500","Open_VIX","Open_MOVE",
        "Drawdown",
        "Momentum_1M","Momentum_3M","Momentum_6M",
        "RV_5d","RV_10d","RV_21d",
        "VIX_Vol_5d","VIX_Vol_10d","VIX_Vol_21d",
        "VIX_Lag1","VIX_Lag2","VIX_Lag5",
        "VIX_MA_5","VIX_MA_10","VIX_MA_20",
        "VIX_STD_5","VIX_STD_10","VIX_Percentile",
        "SPX_Volume_Norm",
        "VIX3M_Spread","VIX_Contango",
        "SPX_Gap","VIX_Gap",
        "VIX_Zscore","VIX_Zscore_Sq","VIX_MeanRev",
        "IV_RV_Ratio","VIX_RV_Spread","VIX_Trend",
        "VIX_MOVE_Ratio","SPX_VIX_Corr_21d","RV_21d_Sq",
        "Open_DXY","Open_GOLD","Open_OIL",
        "Open_HYG","Open_LQD",
        "DXY_overnight","GOLD_overnight","OIL_overnight"
    ]

    dataset_final = dataset[feature_cols + ["Intraday_VIX_Move"]].dropna()

    return dataset_final, dataset

In [5]:
data, dataset =ETL()
data.tail()

,Open_SP500,Open_VIX,Open_MOVE,Drawdown,Momentum_1M,Momentum_3M,Momentum_6M,RV_5d,RV_10d,RV_21d,...,RV_21d_Sq,Open_DXY,Open_GOLD,Open_OIL,Open_HYG,Open_LQD,DXY_overnight,GOLD_overnight,OIL_overnight,Intraday_VIX_Move
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-23,6574.959961,30.040001,108.839996,-0.067653,-0.051795,-0.058368,-0.014874,0.170148,0.145134,0.134177,...,0.018004,99.680000,4353.000000,100.510002,79.300003,108.199997,0.000301,-0.047567,0.022274,2
2026-03-24,6552.089844,25.910000,98.150002,-0.056974,-0.047545,-0.050642,-0.009438,0.177074,0.152945,0.139001,...,0.019321,99.139999,4338.700195,88.779999,79.239998,108.070000,0.001920,-0.014850,0.007375,1
2026-03-25,6598.350098,25.790001,103.010002,-0.060504,-0.041151,-0.054195,-0.015739,0.169109,0.152854,0.135992,...,0.018494,99.199997,4549.000000,88.489998,79.449997,109.000000,-0.002313,0.034028,-0.041798,0
2026-03-26,6555.859863,26.490000,97.589996,-0.055412,-0.043275,-0.048780,-0.014437,0.160078,0.158820,0.134211,...,0.018013,99.629997,4441.500000,91.379997,79.250000,108.279999,0.000301,-0.023803,0.011736,1
2026-03-27,6453.890137,27.540001,115.019997,-0.071854,-0.067515,-0.062061,-0.034873,0.199430,0.163924,0.138824,...,0.019272,99.870003,4371.799805,93.309998,78.839996,107.470001,-0.000300,-0.000846,-0.012384,1


In [6]:
dataset["Intraday_VIX_Move"]. value_counts()

,count
Intraday_VIX_Move,
0,1881
1,1710
2,1629


In [7]:
uri = "blured"
client = MongoClient(uri, tls=True, tlsCAFile=certifi.where())

db = client["DB_VIX"]
collection = db["vix_data"]

data_mongo = data.copy()
data_mongo["_id"] = data_mongo.index.astype(str)  # use datetime as _id

records = data_mongo.to_dict("records")

operations = [
    UpdateOne(
        {"_id": r["_id"]},
        {"$set": r},  # will insert new fields if they exist
        upsert=True
    )
    for r in records
]

if operations:
    result = collection.bulk_write(operations)
    print(f"Inserted or updated records: {result.upserted_count + result.modified_count}")
else:
    print("No new data to insert/update")

Inserted or updated records: 4948
